
### References

*   [https://www.kaggle.com/code/abdmental01/jigsaw-mpnet-base-v2-inference-cv-0-876](https://www.kaggle.com/code/abdmental01/jigsaw-mpnet-base-v2-inference-cv-0-876)
*   [https://www.kaggle.com/code/aerdem4/jigsaw-acrc-qwen7b-finetune-logits-processor-zoo](https://www.kaggle.com/code/aerdem4/jigsaw-acrc-qwen7b-finetune-logits-processor-zoo)
*   [https://www.guruguru.science/competitions/24/discussions/21027ff1-2074-4e21-a249-b2d4170bd516/](https://www.guruguru.science/competitions/24/discussions/21027ff1-2074-4e21-a249-b2d4170bd516/)
*   https://www.kaggle.com/code/mks2192/jigsaw-llama3-1-8b-instruct-training-one-epoch
*   [https://www.kaggle.com/code/fuumin621/qwen2-5-lora-finetune-baseline-inference](https://www.kaggle.com/code/fuumin621/qwen2-5-lora-finetune-baseline-inference)

# 1. Qwen2.5 32B GPTQ Int4 Inference

In [1]:
!uv pip install --system --no-index --find-links='/kaggle/input/jigsaw-packages2/whls/' 'trl==0.21.0' 'optimum==1.27.0' 'auto-gptq==0.7.1' 'bitsandbytes==0.46.1' 'deepspeed==0.17.4' 'logits-processor-zoo==0.2.1' 'vllm==0.10.0'
!uv pip install --system --no-index --find-links='/kaggle/input/jigsaw-packages2/whls/' 'triton==3.2.0'
!uv pip install --system --no-index --find-links='/kaggle/input/jigsaw-packages2/whls/' 'clean-text'
!uv pip install --system --no-index -U --no-deps --find-links='/kaggle/input/jigsaw-packages2/whls/' 'peft' 'accelerate' 'datasets'

Using Python 3.11.11 environment at: /usr
Resolved 168 packages in 640ms
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17

In [2]:
import pandas as pd
import os
all_data=[]

for i in os.listdir('/kaggle/input/negative-subsampling-notebook'):
    if i.endswith('csv'):
        all_data.append(pd.read_csv(f'/kaggle/input/negative-subsampling-notebook/{i}'))

all_data_csv=pd.concat(all_data,axis=0)
all_data_csv.set_index('row_id')
all_data_csv.reset_index(inplace=True,)
all_data_csv['row_id']= all_data_csv.index
all_data_csv.drop('index',inplace=True,axis=1)
all_data_csv.iloc[:].to_csv('full_data.csv',index=False)
all_data_csv.tail()

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,row_id,body,rule,subreddit,positive_example_1,positive_example_2,negative_example_1,negative_example_2,rule_violation
31753,31753,Or those groups that let you clear the entire ...,"No Advertising: Spam, referral links, unsolici...",wow,she will come your home open her legs with an...,code free tyrande --->>> [Imgur](http://i.imgu...,Banks don't want you to know this! Click here ...,SD Stream [ ENG Link 1] (http://www.sportsstre...,NaN
31754,31754,Did you get molested by french people during y...,"No Advertising: Spam, referral links, unsolici...",wow,she will come your home open her legs with an...,code free tyrande --->>> [Imgur](http://i.imgu...,Banks don't want you to know this! Click here ...,SD Stream [ ENG Link 1] (http://www.sportsstre...,NaN
31755,31755,"It seems the mods hate my story about my son, ...","No Advertising: Spam, referral links, unsolici...",wow,she will come your home open her legs with an...,code free tyrande --->>> [Imgur](http://i.imgu...,Banks don't want you to know this! Click here ...,SD Stream [ ENG Link 1] (http://www.sportsstre...,NaN
31756,31756,I did every single quest in first 3 zones got ...,"No Advertising: Spam, referral links, unsolici...",wow,she will come your home open her legs with an...,code free tyrande --->>> [Imgur](http://i.imgu...,Banks don't want you to know this! Click here ...,SD Stream [ ENG Link 1] (http://www.sportsstre...,NaN
31757,31757,Vashj too,"No Advertising: Spam, referral links, unsolici...",wow,she will come your home open her legs with an...,code free tyrande --->>> [Imgur](http://i.imgu...,Banks don't want you to know this! Click here ...,SD Stream [ ENG Link 1] (http://www.sportsstre...,NaN


In [3]:
%%writefile infer_qwen.py

import os
import pandas as pd
from logits_processor_zoo.vllm import MultipleChoiceLogitsProcessor
import torch
import vllm
import numpy as np
from vllm.lora.request import LoRARequest
import argparse
from scipy.special import softmax
df = pd.read_csv("full_data.csv")

MODEL_NAME = "/kaggle/input/qwen2-5-32b-instruct-gptq-int4"
LORA_PATH = "/kaggle/input/qwen2-5-32b-gptq-int4-batch4-full"
if __name__=='__main__':
    os.environ["VLLM_USE_V1"] = "0"

    llm = vllm.LLM(
        MODEL_NAME,
        # quantization='awq',
        quantization='gptq',
        tensor_parallel_size=torch.cuda.device_count(),
        gpu_memory_utilization=0.95,
        trust_remote_code=True,
        dtype="half",
        enforce_eager=True,
        max_model_len=3000,
        disable_log_stats=True,
        enable_prefix_caching=True,
        enable_lora=True,
    )
    tokenizer = llm.get_tokenizer()
    SYS_PROMPT = """
    You are given a comment on reddit. Your task is to classify if it violates the given rule. Only respond Yes/No.
    """
    
    prompts = []
    for i, row in df.iterrows():
        text = f"""
    r/{row.subreddit}
    Rule: {row.rule}
    
    1) {row.positive_example_1}
    Violation: Yes
    
    2) {row.positive_example_2}
    Violation: Yes
    
    3) {row.negative_example_1}
    Violation: No
    
    4) {row.negative_example_2}
    Violation: No
    
    5) {row.body}
    """
        
        messages = [
            {"role": "system", "content": SYS_PROMPT},
            {"role": "user", "content": text}
        ]
    
        prompt = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=False,
        ) + "Answer:"
        prompts.append(prompt)
    
    df["prompt"] = prompts
    
    mclp = MultipleChoiceLogitsProcessor(tokenizer, choices=['Yes','No'])
    outputs = llm.generate(
        prompts,
        vllm.SamplingParams(
            skip_special_tokens=True,
            max_tokens=1,
            logits_processors=[mclp],
            logprobs=2,
        ),
        use_tqdm=True,
        lora_request=LoRARequest("default", 1, LORA_PATH)
    )
    logprobs = [
        {lp.decoded_token: lp.logprob for lp in out.outputs[0].logprobs[0].values()}
        for out in outputs
    ]
    logit_matrix = pd.DataFrame(logprobs)[['Yes','No']]
    df = pd.concat([df, logit_matrix], axis=1)
    
    df[['Yes',"No"]] = df[['Yes',"No"]].apply(lambda x: softmax(x.values), axis=1, result_type="expand")
    df["pred"] = df["Yes"]
    df['rule_violation'] = df["pred"]
    df[['row_id', 'rule_violation']].to_csv("submission_qwen.csv",index=False)
    pd.read_csv('submission_qwen.csv')
    if torch.distributed.is_initialized():
        torch.distributed.destroy_process_group()

Writing infer_qwen.py


In [4]:
!python infer_qwen.py

2025-10-12 07:12:55.963008: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760253176.189030     194 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760253176.255367     194 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
INFO 10-12 07:13:10 [__init__.py:235] Automatically detected platform cuda.
`torch_dtype` is deprecated! Use `dtype` instead!
INFO 10-12 07:13:27 [config.py:1604] Using max model len 3000
WARNING 10-12 07:13:28 [config.py:1084] gptq quantization is not fully optimized yet. The speed can be slower than non-quantized models.
WARNING 10-12 07:13:29 [cuda.py:103] To see benefits of async output processing, enable CUDA graph. Since, enfor

In [5]:
%%writefile infer_llama.py

import os, math, numpy as np
os.environ["CUDA_VISIBLE_DEVICES"]="0,1"

import pandas as pd
import numpy as np

test = pd.read_csv('full_data.csv')
# sub = pd.read_csv('/kaggle/input/jigsaw-agile-community-rules/sample_submission.csv', index_col='row_id')
# sub

if __name__=='__main__':
    import vllm
    
    llm = vllm.LLM(
        "/kaggle/input/jigsaw-llama3-1-8b-instruct-training-one-epoch/llama-8b-instruct-jigsaw",
        tensor_parallel_size=2, 
        gpu_memory_utilization=0.95, 
        trust_remote_code=True,
        dtype="half", 
        enforce_eager=True,
        max_model_len=3000,
        # disable_log_stats=True,
        # enable_prefix_caching=True,
        
    )
    tokenizer = llm.get_tokenizer()
    
    
    from typing import Any, Dict, List
    from transformers import LogitsProcessor
    import torch
    
    choices = ["No", "Yes"]
    
    KEEP = []
    for x in choices:
        c = tokenizer.encode(x,add_special_tokens=False)[0]
        KEEP.append(c)
    print(f"Force predictions to be tokens {KEEP} which are {choices}.")
    
    class DigitLogitsProcessor(LogitsProcessor):
        def __init__(self, tokenizer):
            self.allowed_ids = KEEP
            
        def __call__(self, input_ids: List[int], scores: torch.Tensor) -> torch.Tensor:
            scores[self.allowed_ids] += 100
            return scores
    
    
    
    sys_prompt = '''You are given a comment on reddit and a rule. Your task is to classify whether the comment violates the rule. Only respond Yes/No.'''
    
    
    
    
    
    def formatting(dataset):
        texts = []
        for i in range(len(dataset)):
            texts.append(tokenizer.apply_chat_template(dataset[i], tokenize=False, add_generation_prompt=False))
        return texts
    
    
    
    
    
    template = """
    Subreddit: r/{subreddit}
    Rule: {rule}
    Examples:
    1) {positive_example_1}
    Violation: Yes
    
    2) {negative_example_1}
    Violation: No
    
    3) {negative_example_2}
    Violation: No
    
    4) {positive_example_2}
    Violation: Yes
    Comment:
    {body}
    Violation: """
    
    
    
    dataset = []
    for index,row in test.iterrows():
        
        formatted_sample = [
            {
            "role": "system",
            "content": sys_prompt
        },
           {
               "role": "user",
               "content": template.format(
                   rule = row.rule,
                   subreddit = row.subreddit,
                   body = row.body,
                   positive_example_1 = row.positive_example_1,
                   negative_example_1 = row.negative_example_1,
                   positive_example_2 = row.positive_example_2,
                   negative_example_2 = row.negative_example_2
               )
           }]
        
        dataset.append( formatted_sample )
    
    
    all_prompts = formatting(dataset)
    
    logits_processors = [DigitLogitsProcessor(tokenizer)]
    responses = llm.generate(
        all_prompts,
        vllm.SamplingParams(
            n=1,  # Number of output sequences to return for each prompt.
            top_p=0.9,  # Float that controls the cumulative probability of the top tokens to consider.
            temperature=0,  # randomness of the sampling
            seed=777, # Seed for reprodicibility
            skip_special_tokens=True,  # Whether to skip special tokens in the output.
            max_tokens=1,  # Maximum number of tokens to generate per output sequence.
            logits_processors=logits_processors,
            logprobs = 2
        ),
        use_tqdm = True
    )
    
    results = []
    errors = 0
    
    for i,response in enumerate(responses):
        try:
            x = response.outputs[0].logprobs[0]
            logprobs = []
            for k in KEEP:
                if k in x:
                    logprobs.append( math.exp(x[k].logprob) )
                else:
                    logprobs.append( 0 )
                    print(f"bad logits {i}")
            logprobs = np.array( logprobs )
            logprobs /= logprobs.sum()
            results.append( logprobs )
        except:
            #print(f"error {i}")
            results.append( np.array([1/2., 1/2.]) )
            errors += 1
            
    print(f"There were {errors} inference errors out of {i+1} inferences")
    results = np.vstack(results)
    
    probs = [x[1] for x in results]
    test['rule_violation'] = probs
    test.to_csv('submission_llama.csv')
    if torch.distributed.is_initialized():
        torch.distributed.destroy_process_group()

Writing infer_llama.py


In [6]:
!python infer_llama.py

2025-10-12 08:43:52.149695: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760258632.172404     534 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760258632.180217     534 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
INFO 10-12 08:43:58 [__init__.py:235] Automatically detected platform cuda.
`torch_dtype` is deprecated! Use `dtype` instead!
INFO 10-12 08:44:13 [config.py:1604] Using max model len 3000
WARNING 10-12 08:44:13 [arg_utils.py:1690] Compute Capability < 8.0 is not supported by the V1 Engine. Falling back to V0. 
WARNING 10-12 08:44:14 [cuda.py:103] To see benefits of async output processing, enable CUDA graph. Since, enforce-eager is e

# 3. ENSEMBLE RESULT

In [7]:
import pandas as pd
import numpy as np

q = pd.read_csv('submission_qwen.csv')
l = pd.read_csv('submission_llama.csv')

rq = q['rule_violation'].rank(method='average') / (len(q)+1)
rl = l['rule_violation'].rank(method='average') / (len(l)+1)

blend = 0.6*rq + 0.4*rl   # or tune the rank-weights with a tiny grid using OOF
q['rule_violation'] = blend
q.to_csv('/kaggle/working/submission.csv', index=False)

In [8]:
test= pd.read_csv('full_data.csv')
test['qwen']= rq
test['llama']=rl
test['rule_violation']=blend
test.to_csv('test.csv',index=False)